# Face Attendance Model Architecture

This notebook documents the pretrained face detection and recognition models used by the project. It uses **YuNet** for face detection/alignment landmarks and **SFace** for identity embeddings. No model training or fine-tuning is performed; enrollment stores L2-normalized SFace embeddings as encrypted student templates.

## End-to-end pipeline

```mermaid
flowchart LR
    A[BGR image/frame] --> B[YuNet FaceDetectorYN]
    B --> C{Detected face?}
    C -- No --> D[No face]
    C -- Yes --> E[Align crop with landmarks]
    E --> F[SFace FaceRecognizerSF]
    F --> G[L2-normalized embedding]
    G --> H[Cosine similarity against templates]
    H --> I{Threshold and margins}
    I --> J[Student identity]
    I --> K[Unknown]
    I --> L[Review]
```

In [ ]:
from pathlib import Path
import cv2
import numpy as np

ROOT = Path.cwd()
DETECTOR = ROOT / "models" / "face_detection_yunet_2023mar.onnx"
RECOGNIZER = ROOT / "models" / "face_recognition_sface_2021dec.onnx"

print(f"Detector exists: {DETECTOR.exists()} ({DETECTOR})")
print(f"Recognizer exists: {RECOGNIZER.exists()} ({RECOGNIZER})")

: 

## Model 1: YuNet detector

YuNet is an anchor-free, lightweight face detector that predicts bounding boxes plus five facial landmarks. In this project, OpenCV's `cv2.FaceDetectorYN_create` loads the model and dynamically sets the input size to the source image dimensions.

```mermaid
flowchart TD
    A[Dynamic-size BGR image] --> B[Image pyramid / feature stages]
    B --> C[Multi-scale face predictions]
    C --> D[Bounding boxes]
    C --> E[Five facial landmarks]
    C --> F[Detection confidence]
    D --> G[NMS with configured threshold]
    E --> G
    F --> G
    G --> H[Top-K detected faces]
```

In [ ]:
detector = cv2.FaceDetectorYN_create(
    str(DETECTOR), "", (320, 320),
    score_threshold=0.85, nms_threshold=0.30, top_k=5000
)
print("YuNet detector loaded successfully")

recognizer = cv2.FaceRecognizerSF_create(str(RECOGNIZER), "")
print("SFace recognizer loaded successfully")

## Model 2: SFace recognizer

SFace is used through `cv2.FaceRecognizerSF`. For each accepted detected face, the detector landmarks align and crop the face, the recognizer produces a feature vector, and the project normalizes that vector to unit length. Similarity between two faces is their dot product after normalization (cosine similarity).

```mermaid
flowchart TD
    A[BGR source image] --> B[YuNet box + five landmarks]
    B --> C[alignCrop]
    C --> D[Aligned face tensor]
    D --> E[SFace backbone]
    E --> F[Feature vector]
    F --> G[L2 normalization]
    G --> H[Identity embedding]
```

In [ ]:
def describe_embedding(example_size=(112, 112)):
    # Create a synthetic aligned face for API-level architecture inspection.
    aligned = np.zeros((*example_size, 3), dtype=np.uint8)
    try:
        feature = recognizer.feature(aligned).reshape(-1).astype(np.float32)
        norm = np.linalg.norm(feature)
        normalized = feature / norm if norm else feature
        print("SFace embedding shape:", normalized.shape)
        print("Embedding dtype:", normalized.dtype)
        print("Normalized L2 norm:", float(np.linalg.norm(normalized)))
    except cv2.error as error:
        print("Synthetic-image feature check was not supported by this OpenCV build:", error)

describe_embedding()

## Enrollment and recognition

- Enrollment accepts training images containing exactly one high-quality face.
- Every accepted face is converted to a normalized embedding.
- Templates are stored as a list of embeddings per student in an encrypted gallery.
- Recognition computes cosine similarity between the query embedding and every template of each enrolled identity.
- The highest similarity for each identity is retained, then identities are ranked.
- Low score, uncertain score, or ambiguous top identities are assigned review states.

In [ ]:
def rank_matches(embedding, prototypes):
    embedding = np.asarray(embedding, dtype=np.float32)
    embedding /= np.linalg.norm(embedding)
    scores = []
    for student_id, templates in prototypes.items():
        templates = np.asarray(templates, dtype=np.float32)
        normalized = templates / np.linalg.norm(templates, axis=1, keepdims=True)
        score = float(np.max(normalized @ embedding))
        scores.append((student_id, score))
    return sorted(scores, key=lambda item: item[1], reverse=True)

print("Cosine matching function loaded")

## Quality gates and decision logic

Before matching, `FaceEngine.extract` rejects or marks observations for review using:

| Gate | Default | Purpose |
|---|---:|---|
| Detection confidence | 0.90 minimum | Reject weak detections |
| Face size | 60 pixels minimum | Reject too-small/distant faces |
| Laplacian blur variance | 45.0 minimum | Reject blurred aligned faces |
| Scene detector confidence | 0.85 | YuNet prediction threshold |
| Scene NMS threshold | 0.30 | Suppress overlapping boxes |
| Recognition threshold | Evaluated, fallback 0.363 | Separate known identities from unknowns |
| Review margin | 0.04 | Route borderline cosine scores to review |
| Identity margin | 0.03 | Route ambiguous top candidates to review |

```mermaid
flowchart LR
    A[Face observation] --> B{Quality gates pass?}
    B -- No --> C[Review: quality issue]
    B -- Yes --> D[Cosine rank]
    D --> E{Best score < threshold-margin?}
    E -- Yes --> F[Unknown]
    E -- No --> G{Best score < threshold+margin?}
    G -- Yes --> H[Review: possible identity]
    G -- No --> I{Top scores ambiguous?}
    I -- Yes --> J[Review: ambiguous identities]
    I -- No --> K[Identity]
```

## Configuration charts

The charts below summarize the project thresholds and show how cosine similarity is converted into identity, unknown, and review decisions. The sample ranking is illustrative; runtime rankings come from enrolled embeddings.

![Face attendance architecture charts](reports/model_architecture_charts.png)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import yaml

# Locate the project configuration when the notebook runs from the workspace or repository root.
ROOT = Path.cwd()
while not (ROOT / "config.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

with (ROOT / "config.yaml").open("r", encoding="utf-8") as stream:
    config = yaml.safe_load(stream)

detection = config["detection"]
recognition = config["recognition"]
threshold = float(recognition["initial_threshold"] )
review_margin = float(recognition["review_margin"] )
identity_margin = float(recognition["minimum_identity_margin"] )

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle("Face Attendance Architecture and Configuration", fontsize=16, fontweight="bold")

# 1. Detection configuration
detector_names = ["YuNet confidence", "NMS threshold", "Minimum detection score"]
detector_values = [detection["confidence"], detection["nms_threshold"], detection["minimum_detection_score"]]
bars = axes[0, 0].barh(detector_names, detector_values, color=["#4C78A8", "#F2CF5B", "#59A14F"] )
axes[0, 0].set_xlim(0, 1)
axes[0, 0].set_xlabel("Configured score / threshold")
axes[0, 0].set_title("Detection thresholds")
axes[0, 0].grid(axis="x", alpha=0.25)
for bar, value in zip(bars, detector_values):
    axes[0, 0].text(value + 0.02, bar.get_y() + bar.get_height()/2, f"{value:.2f}", va="center")

# 2. Quality gate values
quality_names = ["Minimum face size (px)", "Minimum blur variance"]
quality_values = [detection["minimum_face_size"], detection["minimum_blur_score"]]
bars = axes[0, 1].bar(quality_names, quality_values, color=["#AF7AA1", "#EDC948"], width=0.55)
axes[0, 1].set_ylabel("Threshold value")
axes[0, 1].set_title("Image quality gates")
axes[0, 1].grid(axis="y", alpha=0.25)
axes[0, 1].tick_params(axis="x", rotation=12)
for bar, value in zip(bars, quality_values):
    axes[0, 1].text(bar.get_x() + bar.get_width()/2, value + 2, f"{value:g}", ha="center")

# 3. Cosine decision bands
low = threshold - review_margin
high = threshold + review_margin
axes[1, 0].axvspan(-1, low, color="#E45756", alpha=0.75, label="Unknown")
axes[1, 0].axvspan(low, high, color="#F2CF5B", alpha=0.85, label="Review")
axes[1, 0].axvspan(high, 1, color="#59A14F", alpha=0.75, label="Identity candidate")
for value, label in [(threshold, "Threshold"), (low, "Threshold - margin"), (high, "Threshold + margin")]:
    axes[1, 0].axvline(value, color="black", linestyle="--", linewidth=1)
    axes[1, 0].text(value, 1.05, label, rotation=45, ha="left", va="bottom", fontsize=8)
axes[1, 0].set_xlim(-1, 1)
axes[1, 0].set_ylim(0, 1)
axes[1, 0].set_yticks([])
axes[1, 0].set_xlabel("Cosine similarity")
axes[1, 0].set_title("Recognition decision bands")
axes[1, 0].legend(loc="lower center", ncols=3, fontsize=8, frameon=False)

# 4. Illustrative identity ranking
sample_scores = {"Student A": 0.72, "Student B": 0.61, "Student C": 0.45, "Student D": 0.32}
student_ids = list(sample_scores.keys())
scores = list(sample_scores.values())
bars = axes[1, 1].barh(student_ids[::-1], scores[::-1], color="#4C78A8")
axes[1, 1].axvline(threshold, color="#E45756", linestyle="--", label=f"Threshold {threshold:.3f}")
axes[1, 1].set_xlim(0, 1)
axes[1, 1].set_xlabel("Illustrative cosine similarity")
axes[1, 1].set_title("Example ranked identity matches")
axes[1, 1].grid(axis="x", alpha=0.25)
axes[1, 1].legend(fontsize=8)
for bar, value in zip(bars, scores[::-1]):
    axes[1, 1].text(value + 0.02, bar.get_y() + bar.get_height()/2, f"{value:.2f}", va="center")

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


## ONNX graph summary

The ONNX files are loaded directly by OpenCV rather than reconstructed with a custom Python network. This keeps the runtime lightweight and ensures the complete project architecture is:

In [ ]:
import yaml

config_path = ROOT / "config.yaml"
with config_path.open("r", encoding="utf-8") as stream:
    config = yaml.safe_load(stream)

print("Project model files:")
for key in ("detector_model", "recognizer_model"):
    print(f"  {key}: {config['paths'][key]}")

print("\nDetection settings:")
for key, value in config["detection"].items():
    print(f"  {key}: {value}")

print("\nRecognition settings:")
for key, value in config["recognition"].items():
    print(f"  {key}: {value}")